# 7주차 : LSTM 모델 완성 및 벤치마크 모델과의 비교

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader

데이터 불러오기

In [2]:
file_path = './processed_sequences/train_data_seq_30d.csv' 
df = pd.read_csv(file_path)

print(f"데이터 전체 형태 {df.shape}")


데이터 전체 형태 (208080, 35)


Train / Test 데이터 분할  
이전 전처리 과정에서 만들어둔 'dataset_type' 컬럼을 활용

In [3]:
train_df = df[df['dataset_type'] == 'Train']
test_df = df[df['dataset_type'] == 'Test']


Feature 와 target 분리  
t-로 시작하는 컬럼들을 찾아 입력피처로 사용

In [4]:
seq_length = 30
feature_cols = [f't-{seq_length - k}' for k in range(seq_length)]

거른 입력 피처들의 열 이름을 통해 값만 불러오기

In [5]:
X_train = train_df[feature_cols].values
y_train = train_df['target'].values

X_test = test_df[feature_cols].values
y_test = test_df['target'].values


In [6]:
print(pd.DataFrame(X_train).head(3))
print(pd.DataFrame(X_train).shape)
print(pd.DataFrame(y_train).shape)
print(pd.DataFrame(X_test).shape)
print(pd.DataFrame(y_test).shape)

         0         1         2         3         4         5         6   \
0  0.179889 -0.022025 -0.022025  1.934757  1.055462 -0.916125  1.939273   
1 -0.022025 -0.022025  1.934757  1.055462 -0.916125  1.939273 -0.507128   
2 -0.022025  1.934757  1.055462 -0.916125  1.939273 -0.507128 -0.022025   

         7         8         9   ...        20        21        22        23  \
0 -0.507128 -0.022025 -0.022025  ...  0.087410 -0.738527 -0.022025 -0.022025   
1 -0.022025 -0.022025 -1.227534  ... -0.738527 -0.022025 -0.022025 -0.422895   
2 -0.022025 -1.227534 -0.453952  ... -0.022025 -0.022025 -0.422895 -1.161767   

         24        25        26        27        28        29  
0 -0.422895 -1.161767  0.238271  0.133356  0.671674 -0.022025  
1 -1.161767  0.238271  0.133356  0.671674 -0.022025 -0.022025  
2  0.238271  0.133356  0.671674 -0.022025 -0.022025 -1.307094  

[3 rows x 30 columns]
(153360, 30)
(153360, 1)
(54720, 30)
(54720, 1)


PyTorch Tensor로 변환 및 차원 추가  
PyTorch LSTM은 항상 3차원 입력을 받는다고 하니 이걸 최대한 구현: (Batch_size, Sequence_length, Input_size)  
논문과 동일하게 하나의 피처(수익률) 사용하므로 Input_size = 1  

In [7]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).unsqueeze(-1) # 마지막 차원 추가
y_train_tensor = torch.tensor(y_train, dtype=torch.long) # 분류를 위한 정수형 타겟

X_test_tensor = torch.tensor(X_test, dtype=torch.float32).unsqueeze(-1)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("\n[데이터 텐서 변환 완료]")
print(f"X_train 형태: {X_train_tensor.shape} -> (배치 크기, 시퀀스 길이={seq_length}, 입력 변수 개수=1)")
print(f"y_train 형태: {y_train_tensor.shape}")

batch_size = 256 # 학습 속도와 안정성을 위한 배치 사이즈 설정

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False) # 테스트 시에는 셔플 생략


[데이터 텐서 변환 완료]
X_train 형태: torch.Size([153360, 30, 1]) -> (배치 크기, 시퀀스 길이=30, 입력 변수 개수=1)
y_train 형태: torch.Size([153360])


## LSTM 셀 만들기

In [8]:
import torch
import torch.nn as nn

class LSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(LSTMCell, self).__init__()
        self.input_size = input_size
        self.hidden_size = hidden_size
        
        self.W_f = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_i = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_s_tilde = nn.Linear(input_size + hidden_size, hidden_size)
        self.W_o = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x_t, hidden_state):
        h_prev, s_prev = hidden_state
        
        combined = torch.cat((x_t, h_prev), dim=1)
        
        f_t = torch.sigmoid(self.W_f(combined))
        i_t = torch.sigmoid(self.W_i(combined))
        s_tilde_t = torch.tanh(self.W_s_tilde(combined))
        s_t = f_t * s_prev + i_t * s_tilde_t
        o_t = torch.sigmoid(self.W_o(combined))
        h_t = o_t * torch.tanh(s_t)
        
        return h_t, s_t

In [9]:
class LSTMModel(nn.Module):
    def __init__(self, input_size=1, hidden_size=25, output_size=2):
        super(LSTMModel, self).__init__()
        self.hidden_size = hidden_size
        
        self.lstm_cell = LSTMCell(input_size, hidden_size)
        self.dropout = nn.Dropout(p=0.1)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        batch_size = x.size(0)
        seq_length = x.size(1)
        
        h_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        s_t = torch.zeros(batch_size, self.hidden_size).to(x.device)
        
        for t in range(seq_length):
            x_t = x[:, t, :] 
            h_t, s_t = self.lstm_cell(x_t, (h_t, s_t))
            
        out = self.dropout(h_t)
        out = self.fc(out)
        
        return out

model = LSTMModel(input_size=1, hidden_size=25, output_size=2)
print("모델 구조:\n", model)

모델 구조:
 LSTMModel(
  (lstm_cell): LSTMCell(
    (W_f): Linear(in_features=26, out_features=25, bias=True)
    (W_i): Linear(in_features=26, out_features=25, bias=True)
    (W_s_tilde): Linear(in_features=26, out_features=25, bias=True)
    (W_o): Linear(in_features=26, out_features=25, bias=True)
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (fc): Linear(in_features=25, out_features=2, bias=True)
)


In [10]:
import torch.optim as optim

In [11]:
criterion = nn.CrossEntropyLoss() 
optimizer = optim.RMSprop(model.parameters(), lr=0.001) 

num_epochs = 10 

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for batch_X, batch_y in train_loader:
        
        optimizer.zero_grad()
        
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
        _, predicted = torch.max(outputs.data, 1) 
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()
        
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.2f}%")

Epoch [1/10], Loss: 0.6649, Accuracy: 61.57%
Epoch [2/10], Loss: 0.6308, Accuracy: 61.09%
Epoch [3/10], Loss: 0.6048, Accuracy: 60.88%
Epoch [4/10], Loss: 0.5807, Accuracy: 60.93%
Epoch [5/10], Loss: 0.5656, Accuracy: 61.24%
Epoch [6/10], Loss: 0.5578, Accuracy: 61.35%
Epoch [7/10], Loss: 0.5534, Accuracy: 61.38%
Epoch [8/10], Loss: 0.5513, Accuracy: 61.50%
Epoch [9/10], Loss: 0.5499, Accuracy: 61.35%
Epoch [10/10], Loss: 0.5489, Accuracy: 61.61%
